# Check a YAML report by hand

Runs `word_input.yaml` through the loader and the docx back end, and shows what
comes out at every stage: the tasks, the finished document, and how it compares
with the reference captured from the old `.rdf`.

## Before you start

Pick the **`.venv313`** kernel of *this* worktree — `Scriptum-Report-dev-yaml`,
branch `dev-yaml`. Its editable install is what makes `import Scriptum` resolve
here rather than to `main`.

If that kernel is not offered, register it once:

```
.venv313/Scripts/python.exe -m pip install ipykernel
.venv313/Scripts/python.exe -m ipykernel install --user --name scriptum-dev-yaml --display-name "Scriptum dev-yaml (3.13)"
```

The next cell checks you are on the right one, so there is no need to guess.


In [1]:
import subprocess
import sys
from pathlib import Path

import Scriptum
import yaml

# Derived from where this notebook sits, so moving the worktree does not
# turn the check below into a false alarm. Jupyter starts in the notebook's
# own directory, and this cell runs before anything changes it.
NOTEBOOK_DIR = Path.cwd()
WORKTREE = NOTEBOOK_DIR.parents[2]
here = Path(Scriptum.__file__).resolve().parent.parent

print('python    ', sys.executable)
print('Scriptum  ', Path(Scriptum.__file__).resolve())
print('PyYAML    ', yaml.__version__)
print('branch    ', subprocess.run(['git', 'branch', '--show-current'], cwd=here,
                                   capture_output=True, text=True).stdout.strip())

if here != WORKTREE:
    print()
    print('WARNING: Scriptum is NOT coming from the dev-yaml worktree.')
    print('         Expected', WORKTREE)
    print('         Pick the .venv313 kernel of that worktree - see the cell above.')
else:
    print()
    print('OK: importing from the dev-yaml worktree.')


python     e:\users\tel\Python\dev\Scriptum-Report-dev-yaml\.venv313\Scripts\python.exe
Scriptum   E:\users\tel\Python\dev\Scriptum-Report-dev-yaml\Scriptum\__init__.py
PyYAML     6.0.3
branch     dev-yaml

OK: importing from the dev-yaml worktree.


## A workspace

The run writes a document and needs its data beside it, so everything is copied
to a temp directory. The repo stays clean, and you can throw the directory away.


In [2]:
import os
import shutil
import tempfile

REPORT_DIR = WORKTREE / 'tests' / '04_examples' / 'wordreport'

def workspace():
    """A fresh directory holding the fixtures, the template and the data."""
    work = Path(tempfile.mkdtemp(prefix='scriptum-'))
    for pattern in ('*.yaml', 'template.docx'):
        for path in REPORT_DIR.glob(pattern):
            shutil.copy(path, work)
    shutil.copytree(REPORT_DIR / 'data', work / 'data', dirs_exist_ok=True)
    os.chdir(work)
    return work

WORK = workspace()
print('working in', WORK)
print(sorted(p.name for p in WORK.iterdir()))


working in C:\Users\tel\AppData\Local\Temp\scriptum-so_ccbvw
['data', 'instruction01.yaml', 'instruction02.yaml', 'preparation01.yaml', 'template.docx', 'testplan01.yaml', 'tool01.yaml', 'word_input.yaml']


## 1. Read the document

`ReportDataFile` reads a `.yaml` document through `Scriptum.rdf.loader`;
anything else is refused with a message. The `.rdf` text parser is gone.

A broken document raises `DocumentError` carrying **every** diagnostic, not the
first — there is a cell near the bottom that shows one.


In [3]:
rdf = Scriptum.ReportDataFile('word_input.yaml')

print('documenttype:', rdf.settings.documenttype)
print('datadir     :', rdf.settings.datadir)
print('tasks       :', len(rdf.tasks))
print('errors      :', rdf.errors or 'none')


documenttype: docx
datadir     : data
tasks       : 75
errors      : none


## 2. What the tasks say

`what` is the operation, `where` the marker an *add* lands at, `target` the
**template name** — the tag written in the .docx — and the address the
**instance**: `subsection:instruction::2` is the second copy of that block.

Note the `_global_` tasks at the end: global fills are applied last, and the
task list carries that rule so no back end has to remember it.


In [4]:
def show_tasks(tasks, limit=None, only=None):
    rows = [t for t in tasks if only is None or only in '.'.join(t.myAddress)]
    print(f'{"#":>4}  {"what":6} {"where":18} {"target":22} address')
    print('-' * 110)
    for t in rows[:limit]:
        print(f'{t.serial:>4}  {t.what or "-":6} {t.where or "-":18} '
              f'{t.target or "-":22} {".".join(t.myAddress)}')
    if limit and len(rows) > limit:
        print(f'... {len(rows) - limit} more')

show_tasks(rdf.tasks, limit=40)


   #  what   where              target                 address
--------------------------------------------------------------------------------------------------------------
   1  apply  -                  -                      section:title::1
   2  -      -                  report:product_name    section:title::1.report:product_name::1
   3  -      -                  report:designer_name   section:title::1.report:designer_name::1
   4  -      -                  report:reason          section:title::1.report:reason::1
   5  -      -                  image:mainmodel        section:title::1.image:mainmodel::1
   6  -      -                  text:description       section:title::1.text:description::1
   7  -      -                  date:creation          section:title::1.date:creation::1
   8  -      -                  author                 section:title::1.:author::1
   9  apply  -                  -                      section:instructions::1
  10  -      -                  text:des

Try `show_tasks(rdf.tasks, only='subsection:instruction')` to see just the
repeated block — instance 1 **applies** to what the template already holds, and
2 and 3 are **copies**.


In [5]:
show_tasks(rdf.tasks, only='subsection:instruction')


   #  what   where              target                 address
--------------------------------------------------------------------------------------------------------------
  16  apply  -                  -                      section:instructions::1.subsection:instruction::1
  17  -      -                  head                   section:instructions::1.subsection:instruction::1.:head::1
  18  -      -                  text:description       section:instructions::1.subsection:instruction::1.text:description::1
  19  add    marker:content::1  image:generic          section:instructions::1.subsection:instruction::1.image:generic::1
  20  copy   -                  -                      section:instructions::1.subsection:instruction::2
  21  -      -                  head                   section:instructions::1.subsection:instruction::2.:head::1
  22  -      -                  text:description       section:instructions::1.subsection:instruction::2.text:description::1
  23  add    mar

## 3. Build the document

Anything the back end could not place prints a `WARNING`. A clean run prints
none.


In [6]:
import contextlib
import io as _io

with contextlib.redirect_stdout(_io.StringIO()) as printed:
    managed = Scriptum.ManagedDocx('template.docx', rdf)
    managed.typesetting(rdf)
    managed.save('report.docx',finish=True)

complaints = [line for line in printed.getvalue().splitlines()
              if 'WARNING' in line or 'ERROR' in line]
print('written:', WORK / 'report.docx')
print('complaints:', len(complaints))
for line in complaints[:20]:
    print('  ', line)


written: C:\Users\tel\AppData\Local\Temp\scriptum-so_ccbvw\report.docx
complaints: 0


## 4. Read it back

Open `report.docx` in Word if you want to look at the formatting; this shows
what it *says*, which is what the automated comparison uses.


In [7]:
import docx

def spoken(path):
    document = docx.Document(path)
    said = [p.text.strip() for p in document.paragraphs]
    for table in document.tables:
        for row in table.rows:
            said.extend(cell.text.strip() for cell in row.cells)
    return [line for line in said if line]

lines = spoken(WORK / 'report.docx')
print(len(lines), 'non-empty lines')
for line in lines[:30]:
    print('  ', line[:100])


213 non-empty lines
   Report ID 4711
   Description
   Testing pudding tasting in space helps researchers understand how microgravity affects the sensory e
   Revisions (of document)
   Test conditions
   These are the general instructions
   Table 1: rocket checks
   Figure 1: prepare a rocket
   Cooking Basics
   You should have at least some knowledge about cooking...
   Cooking Instruction 1
   Get your favorite pudding recipe
   Figure 2: instruction one
   Cooking Instruction 2
   If the first instruction fails, take this one
   Figure 3: instruction two
   Cool it
   Get a fridge, implement it into the rocket, and put the pudding into the fridge
   Figure 4: old fridge
   A refrigerator, commonly shortened to fridge, is a commercial and home appliance consisting of a the
   Figure 5: too cold
   When a pudding gets frozen, the water inside its creamy mixture turns into ice crystals, which expan
   Figure 6: too hot
   When a pudding gets too hot, the delicate balance of its thi

## 5. Compare with the reference

`expected/word_input.json`, beside this notebook, is what this
fixture's **`.rdf`** produced before the back end changed (the `.rdf` and its
parser are gone; the reference is their record). Digits and weekday
names are collapsed on both sides, because the reference was captured on
another day and `date:now` is evaluated per run.

An empty report below means the YAML document says exactly what the text one
said.


In [19]:
import json
import re

DIGITS = re.compile(r'\d+')
WEEKDAY = re.compile(r'\b(?:Mon|Tue|Wed|Thu|Fri|Sat|Sun)\b')

def normalise(lines):
    return [WEEKDAY.sub('#', DIGITS.sub('#', line)) for line in lines]

REFERENCE = REPORT_DIR / 'expected' / 'word_input.json'

expected = normalise(json.loads(REFERENCE.read_text(encoding='utf-8')))
got = normalise(spoken(WORK / 'report.docx'))

print(f'reference {len(expected)} lines, this run {len(got)} lines')
if expected == got:
    print('IDENTICAL')
else:
    for i, (a, b) in enumerate(zip(expected, got)):
        if a != b:
            print(f'first difference at line {i}')
            print('  reference:', a[:110])
            print('  this run :', b[:110])
            break
    else:
        print('one is a prefix of the other')


reference 213 lines, this run 213 lines
IDENTICAL


## 6. Your own document

Edit the string, run the cell, and look at the tasks. Nothing here touches the
repo — it writes into the workspace.

Things worth trying:

- write `subsection:instruction:` twice and watch the ids go `::1`, `::2`
- put an entry under a `marker:content:` and see `what` become `add`
- misspell a namespace and read the diagnostic


In [9]:
SCRATCH = '''
_scriptum_:
  version: 4
  documenttype: docx
  datadir: ./data

_global_:
  report:status: Draft

_content_:
  - section:instructions:
      - text:description: Written by hand
      - subsection:instruction:
          - head: First
          - marker:content:
              - image:generic: {file: recipe1.png, description: one}
      - subsection:instruction:
          - head: Second
'''

(WORK / 'scratch.yaml').write_text(SCRATCH.lstrip(), encoding='utf-8')

scratch = Scriptum.ReportDataFile('scratch.yaml')
show_tasks(scratch.tasks)


   #  what   where              target                 address
--------------------------------------------------------------------------------------------------------------
   1  apply  -                  -                      section:instructions::1
   2  -      -                  text:description       section:instructions::1.text:description::1
   3  apply  -                  -                      section:instructions::1.subsection:instruction::1
   4  -      -                  head                   section:instructions::1.subsection:instruction::1.:head::1
   5  add    marker:content::1  image:generic          section:instructions::1.subsection:instruction::1.image:generic::1
   6  copy   -                  -                      section:instructions::1.subsection:instruction::2
   7  -      -                  head                   section:instructions::1.subsection:instruction::2.:head::1
   8  -      -                  report:status          _global_.report:status


## 7. What a bad document tells you

Errors accumulate: every mistake is reported, not just the first, each with a
file, a line, a column and the path through the document.

The header is validated first and the body only afterwards -- if
``documenttype`` is missing or unknown the walk stops there, because guessing a
namespace ladder would bury that one diagnostic under a page of consequences.
Give it a valid type, as here, and the body reports too.


In [10]:
BROKEN = '''
_scriptum_:
  version: 2
  documenttype: docx
  documentype: docx
  csvseparator: ';;'

_content_:
  - section:a:
      - subsubsection:c:
          - head: too deep for this level
      - width: 4
'''

(WORK / 'broken.yaml').write_text(BROKEN.lstrip(), encoding='utf-8')

from Scriptum.rdf.loader import DocumentError

try:
    Scriptum.ReportDataFile('broken.yaml')
except DocumentError as error:
    print(len(error.diagnostics), 'diagnostics')
    print()
    print(error)


4 diagnostics

broken.yaml:2:12  at _scriptum_
  version is 2; this format needs at least 4
broken.yaml:4:3  at _scriptum_
  unknown setting 'documentype'. Known: csvseparator, datadir, dateformat, datetimeformat, documenttitle, documenttype, floatformat, nvseparator, version
broken.yaml:5:17  at _scriptum_
  csvseparator must be exactly one character, not ';;'
broken.yaml:9:9  at _content_ > section:a > subsubsection:c
  subsubsection:c is at depth 1, where the namespace must be 'subsection', but it has 'subsubsection'. The ladder is: section > subsection > subsubsection > sub3section > sub4section > sub5section. A sequence value is a body, so this reads as a container. A fill's value is a scalar or a mapping.
